# RGT-Est on Google Colab: Poseidon test

Runs the pretrained RGT-Est model (Dou, Wu, Gao & Bi, 2026) on the Poseidon seismic survey.

**Before you start**
1. **Runtime → Change runtime type → GPU** (a T4 is the minimum; pick L4 or A100 if you have access).
2. In Google Drive, make a folder called `RGT-Est` and put these two files in it:
   - `RGT-Est_CIG-Benchmark.pt` (the pretrained model)
   - your Poseidon file, e.g. `Poseidon_part2.npy`

Then run the cells from top to bottom (**Runtime → Run all**).

## 1. Settings: change these if your files are named or placed differently

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/RGT-Est"
MODEL_FILE   = "RGT-Est_CIG-Benchmark.pt"
SEIS_FILE    = "Poseidon_part2.npy"      # or Poseidon_part1.npy
OUTPUT_FILE  = "Poseidon_RGT.npy"         # the result gets saved here in the same folder

## 2. Check the GPU and connect Google Drive

In [ ]:
import torch
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("No GPU found. Go to Runtime → Change runtime type → GPU, then run again.")

from google.colab import drive
drive.mount("/content/drive")

import os
for f in (MODEL_FILE, SEIS_FILE):
    p = os.path.join(DRIVE_FOLDER, f)
    print(("found    " if os.path.exists(p) else "MISSING  ") + p)

## 3. Get the RGT-Est code (your fork)

In [ ]:
!git clone -q https://github.com/fergusrssmith/RGT /content/RGT 2>/dev/null || git -C /content/RGT pull -q
import sys
sys.path.insert(0, "/content/RGT/demo")

## 4. Load and normalise the seismic

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

def normalization(data):
    rng = np.max(data) - np.min(data)
    return (data - np.min(data)) / (rng + 1e-6)

def z_score_clip(data, clp_s=3.0):
    z = (data - np.mean(data)) / (np.std(data) + 1e-8)
    return normalization(np.clip(z, -clp_s, clp_s))

raw = np.load(os.path.join(DRIVE_FOLDER, SEIS_FILE)).astype(np.float32)
print("file shape:", raw.shape)
seis = z_score_clip(raw, clp_s=2).transpose(1, 2, 0) * 2 - 1   # → (inline, xline, time) in [-1, 1]
h, w, t = seis.shape
print("seis shape (inline, xline, time):", seis.shape)

## 5. Run the model

The volume is resized to the fixed grid the model expects (400 × 512 × 512), run through the network, then resized back.
If you get a **CUDA out of memory** error, switch to a bigger GPU (L4/A100) in *Change runtime type*.

In [ ]:
import time
model = torch.jit.load(os.path.join(DRIVE_FOLDER, MODEL_FILE), map_location=device).eval()

infer_shape = (400, 512, 512)
x = torch.from_numpy(seis.transpose(2, 0, 1))[None, None]          # (1, 1, T, H, W)
x = F.interpolate(x, infer_shape, mode="trilinear")
x = torch.cat([x, torch.zeros_like(x), torch.zeros_like(x)], dim=1).to(device)   # no horizon guidance

t0 = time.time()
with torch.no_grad(), torch.autocast(device_type=device):
    rgt = model(nn.ReplicationPad3d(8)(x))[:, :, 8:-8, 8:-8, 8:-8]
print(f"inference took {time.time()-t0:.1f} s")

rgt_volume = F.interpolate(rgt.float(), (t, h, w), mode="trilinear").cpu().numpy()[0, 0]
rgt_volume = z_score_clip(rgt_volume)          # (time, inline, xline), values 0–1
del x, rgt; torch.cuda.empty_cache()

np.save(os.path.join(DRIVE_FOLDER, OUTPUT_FILE), rgt_volume)
print("saved", os.path.join(DRIVE_FOLDER, OUTPUT_FILE), rgt_volume.shape)

## 6. Look at the results

Each row shows a vertical section: the seismic, the predicted RGT (colour = relative age), and the seismic with horizons on top.
The horizons are contour lines of the RGT. If the method works, they should follow the reflectors.

In [ ]:
import matplotlib.pyplot as plt

rgt_hwt = rgt_volume.transpose(1, 2, 0)        # (inline, xline, time), same layout as seis
levels = np.linspace(0.02, 0.98, 35)

def show_sections(axis, n=3):
    idx = np.linspace(0, seis.shape[axis] - 1, n + 2, dtype=int)[1:-1]
    fig, axes = plt.subplots(n, 3, figsize=(16, 3.5 * n))
    for row, i in enumerate(idx):
        s = np.take(seis, i, axis=axis).T          # (time, lateral)
        r = np.take(rgt_hwt, i, axis=axis).T
        v = np.percentile(np.abs(s), 99)
        axes[row, 0].imshow(s, cmap="gray", aspect="auto", vmin=-v, vmax=v)
        axes[row, 1].imshow(r, cmap="jet", aspect="auto")
        axes[row, 2].imshow(s, cmap="gray", aspect="auto", vmin=-v, vmax=v)
        axes[row, 2].contour(r, levels=levels, cmap="jet", linewidths=1.2)
        name = "inline" if axis == 0 else "crossline"
        for ax, title in zip(axes[row], ("seismic", "RGT", "seismic + horizons")):
            ax.set_title(f"{name} {i}: {title}"); ax.set_xlabel("trace"); ax.set_ylabel("time sample")
    fig.tight_layout(); plt.show()

show_sections(axis=0)   # inlines
show_sections(axis=1)   # crosslines

## 7. (Optional) Map view of a few horizons

Traces horizons as surfaces through the whole 3D volume and plots their depth (time) as maps.

In [ ]:
from rgt_helper import horizons_from_rgt

hu = np.linspace(0.15, 0.85, 4)                         # which RGT values to trace
hzs = horizons_from_rgt(sig=1.0, hu=hu, ux=rgt_hwt)     # (n_horizons, inline, xline)

fig, axes = plt.subplots(1, len(hu), figsize=(4.5 * len(hu), 4))
for ax, hz, u in zip(axes, hzs, hu):
    im = ax.imshow(hz, cmap="viridis_r", aspect="auto")
    ax.set_title(f"horizon at RGT={u:.2f}"); ax.set_xlabel("crossline"); ax.set_ylabel("inline")
    plt.colorbar(im, ax=ax, label="time sample")
fig.tight_layout(); plt.show()